# ED Triage AI — Data Cleaning Notebook

Cleans the consolidated ED triage dataset for downstream feature engineering and model training.

**Input:** `s3://ed-triage-capstone-group7/Data_Output/consolidated_dataset_PMH_v2.csv` (22 columns × 9,149 rows)  
**Output:** `s3://ed-triage-capstone-group7/Data_Output/consolidated_dataset_cleaned.csv` (cleaned, 8,383 rows)

| Section | Column | Operation |
|---------|--------|----------|
| 1 | — | Load data |
| 2 | — | Drop excluded columns |
| 3 | `chiefcomplaint` | Underscore strip, noise removal, abbreviation expansion |
| 4 | `HPI` | `[REDACTED]` replacement, whitespace normalization |
| 5 | `past_medical_history` | `[REDACTED]` replacement, null→empty string |
| 6 | `initial_vitals` | Parse to 6 float columns, range validation |
| 7 | `patient_info` | Parse age/gender/race, normalize categories |
| 8 | `pain` | Ceiling rounding, range strings, semantic mapping |
| 9 | `arrival_transport` | Uppercase, OTHER→UNKNOWN |
| 10 | — | Drop null rows |
| 11 | — | Save cleaned dataset |

## Section 1 — Load Data

Import required libraries, load the consolidated dataset, and inspect its structure.


In [59]:
import os
import io
import re
import math
import warnings

import boto3
import pandas as pd
import numpy as np
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ------------
# AWS Config
# ------------
AWS_PROFILE = os.getenv("AWS_PROFILE", "ed-triage")
AWS_REGION  = os.getenv("AWS_REGION",  "us-east-1")
S3_BUCKET   = os.getenv("S3_BUCKET",   "ed-triage-capstone-group7")

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3      = session.client("s3")

def read_csv_from_s3(key: str) -> pd.DataFrame:
    """Download a CSV from S3 and return it as a DataFrame."""
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()))

# ---------------------------------------------------------------------------
# Load the consolidated dataset produced by EDA.ipynb
# ---------------------------------------------------------------------------
INPUT_KEY = "Data_Output/consolidated_dataset_PMH_v2.csv"

df = read_csv_from_s3(INPUT_KEY)
print(f"Loaded from : s3://{S3_BUCKET}/{INPUT_KEY}")
print(f"Shape       : {df.shape}")
print()
df.info()

Loaded from : s3://ed-triage-capstone-group7/Data_Output/consolidated_dataset_PMH_v2.csv
Shape       : (9149, 22)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9149 entries, 0 to 9148
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   stay_id               9149 non-null   int64 
 1   subject_id            9149 non-null   int64 
 2   hadm_id               9149 non-null   int64 
 3   text                  9149 non-null   object
 4   HPI                   9148 non-null   object
 5   patient_info          9149 non-null   object
 6   initial_vitals        8779 non-null   object
 7   tests                 9149 non-null   object
 8   past_medication       8632 non-null   object
 9   diagnosis             9149 non-null   object
 10  primary_diagnosis     9149 non-null   object
 11  secondary_diagnosis   9149 non-null   object
 12  triage                9149 non-null   int64 
 13  pain                  8

In [60]:
# Preview first 3 rows to understand raw data structure
df.head(3)


,stay_id,subject_id,hadm_id,text,HPI,patient_info,initial_vitals,tests,past_medication,diagnosis,...,triage,pain,chiefcomplaint,arrival_transport,disposition,icd_code,icd_title,icd_version,specialty,past_medical_history
0,32822973,10002428,20321825,<br>Name: ___ ___ No: ___\n\n...,Patient is an ___ year-old patient with histor...,"Gender: Female, Race: WHITE, Age: 80","Temperature: 100.2, Heartrate: 95.0, resprate:...",\nlabs upon admission:\n___ 07:30am blood wbc-...,\nmedications at ___:\nfluticasone 50 mcg nasa...,\nPrimary diagnoses:\nAltered mental status\n...,...,2,unable,LETHARGY/SOB,AMBULANCE,ADMITTED,51881,ACUTE RESPIRATORY FAILURE,9,"['Neurology', 'Pulmonology']",Anemia\nBorderline cholesterol\nC. Diff\nFlatu...
1,31591237,10003019,24646702,<br>Name: ___. Unit No: ___\n\...,___ year old male with history to intestinal s...,"Gender: Male, Race: WHITE, Age: 69","Temperature: 97.8, Heartrate: 87.0, resprate: ...",\n___ 05:21am blood wbc-6.3 rbc-3.17* hgb-8.4*...,\nprednisone 40'\nalendronate 70'\nazathioprin...,\nPerforated Duodenal Ulcer\n\n\n,...,2,10,SEVERE ABD PAIN,WALK IN,ADMITTED,78909,ABDOMINAL PAIN OTHER SPECIED,9,['Gastroenterology'],1. Sarcoidosis<comma> dx skin bx: intestinal &...
2,36819102,10004719,21197153,<br>Name: ___ Unit No: ___\...,___ w Rt AK pop to ___ bypass with NRGSV for a...,"Gender: Female, Race: WHITE, Age: 66","Temperature: 98.7, Heartrate: 93.0, resprate: ...",\n___ 05:45am blood wbc-9.0 rbc-3.91 hgb-11.5 ...,\nthe preadmission medication list is accurate...,\nPeripheral Arterial Disease\nRight Posterio...,...,3,5,R Leg pain,WALK IN,ADMITTED,I82441,Acute embolism and thrombosis of right tibial ...,10,"['Vascular Surgery', 'Vascular Surgery']",PMH: DVT R pop v (___)<comma> asthma<comma> Rt...


## Section 2 — Drop Excluded Columns

Retain only the columns relevant to triage modeling. All other columns are dropped to prevent leakage.

**Dropped:** `subject_id`, `hadm_id`, `text`, `tests`, `past_medication`, `diagnosis`, `primary_diagnosis`, `secondary_diagnosis`, `disposition`, `icd_code`, `icd_title`, `icd_version`, `specialty`


In [61]:
# Columns required for downstream modeling and text processing
KEEP_COLUMNS = [
    'stay_id',              # row identifier — not a model feature
    'triage',               # target variable — do not modify
    'chiefcomplaint',
    'HPI',
    'past_medical_history',
    'initial_vitals',
    'patient_info',
    'pain',
    'arrival_transport',
]

# Drop all columns not in KEEP_COLUMNS, preserving column order
df = df[KEEP_COLUMNS]

# Confirm exact column set and order after drop
assert df.columns.tolist() == KEEP_COLUMNS, (
    f'Column mismatch. Got: {df.columns.tolist()}'
)

print(f'Shape after column drop: {df.shape}')
print()
print('Null counts per column:')
print(df.isnull().sum())


Shape after column drop: (9149, 9)

Null counts per column:
stay_id                   0
triage                    0
chiefcomplaint            2
HPI                       1
past_medical_history    292
initial_vitals          370
patient_info              0
pain                    260
arrival_transport         0
dtype: int64


## Section 3 — Clean: `chiefcomplaint`

Four-step cleaning pipeline:

1. **Underscore placeholders** — values that are entirely underscores/spaces → `NaN`; underscores mixed with real text are stripped.
2. **Noise characters** — remove `?`; remove standalone `+` not used in clinical shorthand (e.g. `HIV+`, `COVID+`).
3. **Abbreviation expansion** — whole-word, case-insensitive substitution using a curated clinical abbreviation map.
4. **Final null check** — empty/whitespace values after cleaning → `NaN`.


In [62]:
# Snapshot null count before any cleaning
null_before_cc = df['chiefcomplaint'].isna().sum()
print(f'Null count BEFORE cleaning: {null_before_cc}')
print(f'Total unique values: {df["chiefcomplaint"].nunique()}')

Null count BEFORE cleaning: 2
Total unique values: 2271


In [63]:
# Step 1: Handle underscore placeholders
# MIMIC uses sequences of underscores as PHI redaction markers.
# Entire values that are only underscores/spaces -> NaN.
# Underscores mixed with real text -> strip underscores, preserve content.

def clean_underscores(val):
    if not isinstance(val, str):
        return val
    # Entire value is only underscores and/or spaces → null
    if re.fullmatch(r'[_ ]+', val):
        return None
    # Mixed value: remove underscores, keep real text
    return val.replace('_', '')

df['chiefcomplaint'] = df['chiefcomplaint'].apply(clean_underscores)
print(f'After Step 1 (underscore handling) — nulls: {df["chiefcomplaint"].isna().sum()}')


After Step 1 (underscore handling) — nulls: 7


In [64]:
# Step 2: Remove noise characters
# Remove '?' (data-entry artefact). Remove standalone '+' that is not part of
# a clinical positive marker such as HIV+, COVID+
# Chain multiple fixed-width negative lookbehinds (Python re requires fixed width).

STANDALONE_PLUS_RE = re.compile(
    r'(?<!HIV)(?<!COVID)(?<!HCV)(?<!HBV)(?<!MRSA)\+',
    re.IGNORECASE,
)

def remove_noise(val):
    if not isinstance(val, str):
        return val
    val = val.replace('?', '')             # remove question marks
    val = STANDALONE_PLUS_RE.sub('', val)  # remove standalone '+'
    return val.strip()

df['chiefcomplaint'] = df['chiefcomplaint'].apply(remove_noise)
print(f'After Step 2 (noise removal) — nulls: {df["chiefcomplaint"].isna().sum()}')

After Step 2 (noise removal) — nulls: 7


In [65]:
# Step 3: Expand clinical abbreviations
# Whole word and case insensitive replacements 
# Each abbreviation is applied as a separate substitution to avoid partial matches.

ABBREVIATION_MAP = {
    'brbpr':   'bright red blood per rectum',
    'sbo':     'small bowel obstruction',
    'n/v':     'nausea and vomiting',
    'n/v/d':   'nausea vomiting and diarrhea',
    'sob':     'shortness of breath',
    'cp':      'chest pain',
    'ha':      'headache',
    'abd':     'abdominal',
    'htn':     'hypertension',
    'dm':      'diabetes mellitus',
    'uti':     'urinary tract infection',
    'ams':     'altered mental status',
    'loc':     'loss of consciousness',
    'mva':     'motor vehicle accident',
    'gi':      'gastrointestinal',
    'gu':      'genitourinary',
    'ms':      'mental status',
    'lt':      'left',
    'rt':      'right',
    'bilat':   'bilateral',
    'w/':      'with',
    's/p':     'status post',
    'h/o':     'history of',
    'f/u':     'follow up',
    'r/o':     'rule out',
    'c/o':     'complaint of',
    'wt':      'weight',
    'wt loss': 'weight loss',
}

def expand_abbreviations(val):
    if not isinstance(val, str):
        return val
    for abbrev, expansion in ABBREVIATION_MAP.items():
        # re.escape handles slashes and special regex chars in abbreviation keys
        pattern = r'(?i)\b' + re.escape(abbrev) + r'\b'
        val = re.sub(pattern, expansion, val)
    return val

df['chiefcomplaint'] = df['chiefcomplaint'].apply(expand_abbreviations)
print(f'After Step 3 (abbreviation expansion) — nulls: {df["chiefcomplaint"].isna().sum()}')


After Step 3 (abbreviation expansion) — nulls: 7


In [66]:
# Step 4: Final null check — empty/whitespace strings after cleaning -> NaN
# the +5 additions are result of __ converting to NaN

def nullify_empty(val):
    if isinstance(val, str) and val.strip() == '':
        return None
    return val

df['chiefcomplaint'] = df['chiefcomplaint'].apply(nullify_empty)

null_after_cc = df['chiefcomplaint'].isna().sum()
print(f'Null count BEFORE cleaning : {null_before_cc}')
print(f'Null count AFTER  cleaning : {null_after_cc}')
print(f'Net change                 : {null_after_cc - null_before_cc:+d}')
print()
print('── 5 sample unique values after cleaning ──')
samples = df['chiefcomplaint'].dropna().unique()[:5]
for i, s in enumerate(samples, 1):
    print(f'  [{i}] {s}')


Null count BEFORE cleaning : 2
Null count AFTER  cleaning : 7
Net change                 : +5

── 5 sample unique values after cleaning ──
  [1] LETHARGY/shortness of breath
  [2] SEVERE abdominal PAIN
  [3] R Leg pain
  [4] L KNEE PAIN
  [5] abdominal pain


## Section 4 — Clean: `HPI`

**Goal:** Preserve the clinical narrative while removing MIMIC de-identification artifacts.

Steps:
1. Replace 2+ consecutive underscores with `[REDACTED]`
2. Normalize whitespace (collapse multiples, strip edges)
3. Set empty/whitespace-only values to `NaN`


In [67]:
# Null count before cleaning
hpi_null_before = df['HPI'].isna().sum()
print(f'Null count BEFORE cleaning: {hpi_null_before}')

def clean_hpi(text):
    """
    Clean a single HPI string:
    - Replace MIMIC de-id placeholders (2+ underscores) with [REDACTED]
    - Collapse multiple spaces; strip leading/trailing whitespace
    - Return NaN if result is empty or whitespace-only
    """
    if not isinstance(text, str):
        return float('nan')
    # Step 1: Replace 2+ underscores (MIMIC de-id artifacts) with [REDACTED]
    text = re.sub(r'_{2,}', '[REDACTED]', text)
    # Step 2: Normalize whitespace
    text = re.sub(r' {2,}', ' ', text).strip()
    # Step 3: Return NaN if nothing meaningful remains
    if not text:
        return float('nan')
    return text

df['HPI'] = df['HPI'].apply(clean_hpi)

hpi_null_after = df['HPI'].isna().sum()
print(f'Null count AFTER cleaning:  {hpi_null_after}')
print()
print('Sample HPI entries (up to 200 chars each):')
for i, val in enumerate(df['HPI'].dropna().head(2), start=1):
    print(f'  [{i}] {str(val)[:200]}')


Null count BEFORE cleaning: 1
Null count AFTER cleaning:  1

Sample HPI entries (up to 200 chars each):
  [1] Patient is an [REDACTED] year-old patient with history of Sjogren's
syndrome, moderate MR, recent hospitalization for sepsis
secondary to c. diff colitis complicated by hypercarbic
respiratory failure
  [2] [REDACTED] year old male with history to intestinal sarcoidosis s/p
Ex.laparoscopy, right colectomy [REDACTED] ([REDACTED]), ITP s/p open
splenectomy, and inguinal hernia repair complaining of new
abd


## Section 5 — Clean: `past_medical_history`

**Goal:** Light cleaning only. This column is a list of prior conditions — structure must be preserved.

Null values are set to `""` (empty string), **not** dropped. Missing PMH means no recorded prior history and is clinically meaningful; the downstream text builder will omit the PMH clause rather than treating it as data loss.


In [68]:
# Null count before cleaning
pmh_null_before = df['past_medical_history'].isna().sum()
print(f'Null count BEFORE cleaning: {pmh_null_before}')

def clean_past_medical_history(text):
    """
    Light cleaning for past_medical_history:
    - Replace MIMIC de-id placeholders (2+ underscores) with [REDACTED]
    - Normalize whitespace
    - NaN values → empty string (no PMH on record; do not drop the row)
    """
    if not isinstance(text, str):
        return ''
    text = re.sub(r'_{2,}', '[REDACTED]', text)
    text = re.sub(r' {2,}', ' ', text).strip()
    return text

df['past_medical_history'] = df['past_medical_history'].apply(clean_past_medical_history)

pmh_null_after = df['past_medical_history'].isna().sum()
print(f'Null count AFTER cleaning:  {pmh_null_after}')
print(f'Empty-string count:         {(df["past_medical_history"] == "").sum()}')
print()
print('Sample past_medical_history values (5 entries):')
for i, val in enumerate(df['past_medical_history'].head(5), start=1):
    print(f'  [{i}] {repr(val)}')


Null count BEFORE cleaning: 292
Null count AFTER cleaning:  0
Empty-string count:         292

Sample past_medical_history values (5 entries):
  [1] 'Anemia\nBorderline cholesterol\nC. Diff\nFlatulence\nHealth Maintenance\nHeart Murmur\nHypertension\nHypothyroidism\nMitral Regurgitation\nOsteoporosis\nPneumonia\nSinusitis\n[REDACTED]'
  [2] '1. Sarcoidosis<comma> dx skin bx: intestinal & pulmonary involvement<comma>\nrecurrent iritis\n2. Inflammatory bowel disease; s/p ileo-hemicolectomy [REDACTED]<comma>\npath +sarcoid\n3. GERD.\n4. Hyperlipidemia\n5 OSA on CPAP\n6. Asthma.\n7. Osteoarthritis.\n8. Fractured pelvis<comma> [REDACTED] s/p fall.\n9. BPH<comma> status post prostatectomy.\n10. Depression.\n11. History of ITP<comma> status post splenectomy in [REDACTED].\n12. Hard of hearing and wears hearing aid.'
  [3] 'PMH: DVT R pop v ([REDACTED])<comma> asthma<comma> Rt pop artery thrombus with\nnegative hypercoagulable workup'
  [4] '1. Hypertension\n2. Hypothyroidism<comma> status pos

## Section 6 — Clean: `initial_vitals`

**Raw format:** `"Temperature: 100.2, Heartrate: 95.0, resprate: 28.0, o2sat: 100.0, sbp: 99.0, dbp: 47.0"`

Steps:
1. Parse raw string into 6 float columns: `temp_f`, `heart_rate`, `resp_rate`, `spo2`, `sbp`, `dbp`
2. Add `vitals_any_missing` flag — `True` if any of the 6 parsed vitals is NaN
3. Validate clinical ranges — flag out-of-range rows in `vitals_out_of_range` 
4. Null summary per vital column

In [69]:
def parse_vitals(s):
    """
    Extract 6 vital-sign values from a comma-separated text string.
    Returns a dict with float values; missing fields map to NaN.
    """
    if pd.isna(s):
        return {c: np.nan for c in ['temp_f', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'dbp']}
    patterns = {
        'temp_f':     r'[Tt]emperature[:\s]+([\d.]+)',
        'heart_rate': r'[Hh]eart[Rr]ate[:\s]+([\d.]+)',
        'resp_rate':  r'[Rr]esp[Rr]ate[:\s]+([\d.]+)',
        'spo2':       r'[Oo]2[Ss]at[:\s]+([\d.]+)',
        'sbp':        r'\bsbp[:\s]+([\d.]+)',
        'dbp':        r'\bdbp[:\s]+([\d.]+)',
    }
    result = {}
    for col, pat in patterns.items():
        m = re.search(pat, s)
        result[col] = float(m.group(1)) if m else np.nan
    return result

# Apply parser and join resulting columns back onto df
vitals_parsed = df['initial_vitals'].apply(parse_vitals).apply(pd.Series)
df = pd.concat([df, vitals_parsed], axis=1)

print(f'Shape after parsing initial_vitals: {df.shape}')
print('\nNew vital columns added:', list(vitals_parsed.columns))
print('\nSample parsed rows (3):')
print(df[['initial_vitals', 'temp_f', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'dbp']].head(3).to_string())


Shape after parsing initial_vitals: (9149, 15)

New vital columns added: ['temp_f', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'dbp']

Sample parsed rows (3):
                                                                            initial_vitals  temp_f  heart_rate  resp_rate   spo2    sbp   dbp
0  Temperature: 100.2, Heartrate: 95.0, resprate: 28.0, o2sat: 100.0, sbp: 99.0, dbp: 47.0   100.2        95.0       28.0  100.0   99.0  47.0
1   Temperature: 97.8, Heartrate: 87.0, resprate: 26.0, o2sat: 99.0, sbp: 119.0, dbp: 63.0    97.8        87.0       26.0   99.0  119.0  63.0
2   Temperature: 98.7, Heartrate: 93.0, resprate: 16.0, o2sat: 99.0, sbp: 166.0, dbp: 97.0    98.7        93.0       16.0   99.0  166.0  97.0


In [70]:
vital_cols = ['temp_f', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'dbp']

# Flag rows where ANY of the 6 parsed vitals is NaN.
# For this project, we decided "missingness" is clinically meaningful (patient uncooperative, field crew didn't measure, etc.)

df['vitals_any_missing'] = df[vital_cols].isna().any(axis=1)

print(f'vitals_any_missing=True  : {df["vitals_any_missing"].sum()}')
print(f'vitals_any_missing=False : {(~df["vitals_any_missing"]).sum()}')

# Validate vital ranges — flag outliers
# Ranges based on clinical literature
VITAL_RANGES = {
    'temp_f':     (80,  115),
    'heart_rate': (10,  300),
    'resp_rate':  (1,    60),
    'spo2':       (50,  100),
    'sbp':        (40,  300),
    'dbp':        (20,  200),
}

# Build boolean mask: True if any non-null vital falls outside its valid range
out_of_range_mask = pd.Series(False, index=df.index)
for col, (lo, hi) in VITAL_RANGES.items():
    col_mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    out_of_range_mask = out_of_range_mask | col_mask

df['vitals_out_of_range'] = out_of_range_mask

flagged_count = df['vitals_out_of_range'].sum()
print(f'\nRows flagged with at least one vital out of range: {flagged_count}')
print('\nPer-vital out-of-range breakdown:')
for col, (lo, hi) in VITAL_RANGES.items():
    n = (df[col].notna() & ((df[col] < lo) | (df[col] > hi))).sum()
    print(f'  {col:12s}  range [{lo}, {hi}]  flagged: {n}')

vitals_any_missing=True  : 784
vitals_any_missing=False : 8365

Rows flagged with at least one vital out of range: 34

Per-vital out-of-range breakdown:
  temp_f        range [80, 115]  flagged: 8
  heart_rate    range [10, 300]  flagged: 1
  resp_rate     range [1, 60]  flagged: 0
  spo2          range [50, 100]  flagged: 5
  sbp           range [40, 300]  flagged: 11
  dbp           range [20, 200]  flagged: 14


In [71]:
# Null summary for parsed vital columns
# 370 rows with all-NaN vitals correspond to records where initial_vitals was null

vital_cols = ['temp_f', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'dbp']

null_summary = df[vital_cols].isna().sum().rename('null_count').to_frame()
null_summary['null_pct'] = (null_summary['null_count'] / len(df) * 100).round(2)
print('Null counts per parsed vital column:')
print(null_summary.to_string())

all_null_vitals = df[vital_cols].isna().all(axis=1).sum()
print(f'\nRows where ALL 6 vitals are NaN: {all_null_vitals}')

Null counts per parsed vital column:
            null_count  null_pct
temp_f             581      6.35
heart_rate         397      4.34
resp_rate          513      5.61
spo2               484      5.29
sbp                408      4.46
dbp                425      4.65

Rows where ALL 6 vitals are NaN: 370


## Section 7 — Clean: `patient_info`

**Raw format:** `"Gender: Female, Race: WHITE, Age: 80"`

Steps:
1. Parse into 3 typed columns: `age` (int), `gender` (string), `race` (string)
2. Normalise `gender` → `Female` / `Male` / `Unknown`
3. Normalise `race` into 5 groups: `WHITE`, `BLACK`, `HISPANIC`, `ASIAN`, `OTHER`
4. Validate age range (0–120); flag but do not drop


In [72]:
def parse_patient_info(s):
    """
    Extract gender, race, and age from a structured text string.
    Returns a dict; missing fields map to NaN.
    """
    if pd.isna(s):
        return {'age': np.nan, 'gender': np.nan, 'race': np.nan}
    gender_m = re.search(r'Gender:\s*([^,]+)', s)
    race_m   = re.search(r'Race:\s*([^,]+)', s)
    age_m    = re.search(r'Age:\s*(\d+)', s)
    return {
        'gender': gender_m.group(1).strip() if gender_m else np.nan,
        'race':   race_m.group(1).strip()   if race_m   else np.nan,
        'age':    int(age_m.group(1))        if age_m    else np.nan,
    }

demo_parsed = df['patient_info'].apply(parse_patient_info).apply(pd.Series)
df['age']    = demo_parsed['age']
df['gender'] = demo_parsed['gender']
df['race']   = demo_parsed['race']

print(f'Shape after parsing patient_info: {df.shape}')
print('\nNull counts — age, gender, race:')
print(df[['age', 'gender', 'race']].isna().sum().to_string())


Shape after parsing patient_info: (9149, 20)

Null counts — age, gender, race:
age       0
gender    0
race      0


In [73]:
# Normalise gender -> Female / Male / Unknown
# Collapse all values to three canonical labels via case-insensitive matching.

def normalise_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_lower = str(g).strip().lower()
    if g_lower == 'female':
        return 'Female'
    elif g_lower == 'male':
        return 'Male'
    else:
        return 'Unknown'

df['gender'] = df['gender'].apply(normalise_gender)

print('Gender value counts after normalisation:')
print(df['gender'].value_counts(dropna=False).to_string())


Gender value counts after normalisation:
gender
Female    4701
Male      4448


In [74]:
# Normalise race -> 5 canonical groups
# selected/generalized groups. Any other value becomes OTHER.

def map_race(r):
    if pd.isna(r):
        return 'OTHER'
    r_up = r.upper()
    if 'WHITE' in r_up:
        return 'WHITE'
    elif 'BLACK' in r_up or 'AFRICAN' in r_up:
        return 'BLACK'
    elif 'HISPANIC' in r_up or 'LATINO' in r_up:
        return 'HISPANIC'
    elif 'ASIAN' in r_up:
        return 'ASIAN'
    else:
        return 'OTHER'

# Print full audit table before applying (shows every unique raw -> mapped group)
raw_race_unique = df['race'].value_counts(dropna=False).reset_index()
raw_race_unique.columns = ['raw_value', 'count']
raw_race_unique['mapped_group'] = raw_race_unique['raw_value'].apply(map_race)
print('Race mapping audit (all unique raw values → mapped group):')
print(raw_race_unique.to_string(index=False))

df['race'] = df['race'].apply(map_race)
print('\nRace value counts after normalisation:')
print(df['race'].value_counts(dropna=False).to_string())


Race mapping audit (all unique raw values → mapped group):
                                raw_value  count mapped_group
                                    WHITE   6080        WHITE
                   BLACK/AFRICAN AMERICAN    920        BLACK
                                    OTHER    375        OTHER
           HISPANIC/LATINO - PUERTO RICAN    225     HISPANIC
                   WHITE - OTHER EUROPEAN    219        WHITE
                                  UNKNOWN    185        OTHER
                          ASIAN - CHINESE    133        ASIAN
                                    ASIAN    118        ASIAN
                          WHITE - RUSSIAN    118        WHITE
              HISPANIC/LATINO - DOMINICAN    114     HISPANIC
                       BLACK/CAPE VERDEAN    110        BLACK
                       HISPANIC OR LATINO     85     HISPANIC
                            BLACK/AFRICAN     59        BLACK
                   BLACK/CARIBBEAN ISLAND     54        BLACK
           

In [75]:
# Validate age range — flag but do not drop
# Values outside 0–120 indicate data entry errors or unit mismatch.

age_out_of_range = df['age'].notna() & ((df['age'] < 0) | (df['age'] > 120))
print(f'Rows with age outside 0-120: {age_out_of_range.sum()}')

if age_out_of_range.sum() > 0:
    print('\nFlagged age values:')
    print(df.loc[age_out_of_range, ['stay_id', 'age', 'patient_info']].to_string(index=False))

print('\nAge descriptive stats:')
print(df['age'].describe().round(1).to_string())

print('\nSample rows — parsed patient_info fields (3):')
print(df[['patient_info', 'gender', 'race', 'age']].head(3).to_string())


Rows with age outside 0-120: 0

Age descriptive stats:
count    9149.0
mean       56.7
std        19.7
min        18.0
25%        42.0
50%        58.0
75%        72.0
max        91.0

Sample rows — parsed patient_info fields (3):
                           patient_info  gender   race  age
0  Gender: Female, Race: WHITE, Age: 80  Female  WHITE   80
1    Gender: Male, Race: WHITE, Age: 69    Male  WHITE   69
2  Gender: Female, Race: WHITE, Age: 66  Female  WHITE   66


## Section 8 — Clean: `pain`

Convert all pain values to a numeric score 0–10 using context-aware mapping.

**Pipeline:**
1. Add `pain_missing` flag **before** any modification (preserves original missingness signal)
2. Parse clean numeric values (float → int)
3. Round decimal values using ceiling (`7.5` → `8`)
4. Handle range strings (`"5-6"` → `6`, take the higher value)
5. Apply semantic mapping for text descriptors with clinical rationale
6. Final validation and distribution audit


In [76]:
# Step 1: Add pain_missing flag BEFORE any modification
# Flag rows where pain is null OR cannot be parsed as a valid 0-10 numeric score.
# This preserves the original missingness signal even after imputation.

def is_numeric_pain(val):
    """Return True only if val parses as a float in [0, 10]."""
    try:
        v = float(str(val).strip())
        return 0 <= v <= 10
    except:
        return False

df['pain_missing'] = (~df['pain'].apply(is_numeric_pain)).astype(int)
print('pain_missing=1 count:', df['pain_missing'].sum())
print('pain_missing=0 count:', (df['pain_missing'] == 0).sum())


pain_missing=1 count: 1014
pain_missing=0 count: 8135


In [77]:
# Steps 2-7: Convert pain to numeric 0-10
# Process order:
#   1. Clean floats in [0,10]
#   2. Pre-process: strip surrounding quotes and trailing punctuation
#   3. Out-of-range numerics (>10) → cap at 10
#   4. Prehospital tokens -> NaN
#   5. X/Y slash format (e.g. '3/10', '8/10', '10/') → parse numerator
#   6. Range strings (e.g. '5-6') -> take higher value
#   7. Semantic map
# Every non-numeric raw value is tracked in audit_records for the audit table.

PAIN_MAP = {
    # Unknown / unassessable — sentinel 0, flagged via pain_missing=1
    'unknown': 0, 'uta': 0, 'ua': 0, 'unable to assess': 0, 'unable to obtain': 0,
    'u/a': 0, 'n/a': 0, 'na': 0,
    'unable': 0, 'u': 0,          # common abbreviations missing from original map
    '?': 0,                        # unknown, flagged
    # Explicitly no pain
    'none': 0, 'no pain': 0, 'denies pain': 0, 'no': 0,
    # Mild / managed
    'mild': 2, 'a little': 2, 'a little bit': 2, 'slight': 2, 'minor': 2, 'low': 2,
    'controlled': 2, 'ok': 2, 'vague': 2,
    # Mild-moderate
    'not too bad': 3, 'not bad': 3, 'not so bad': 3, 'discomfort': 3,
    # Moderate
    'moderate': 5, 'some pain': 5, 'uncomfortable': 5, 'mod': 5,
    'hurts': 5, 'cramping': 5,    # unscored but clearly present
    # Severe
    'a lot': 8, 'alot': 8,        # alot is a common spelling variant
    'significant': 8, 'severe': 8, 'very painful': 8, 'very bad': 8, 'high': 8, 'bad': 8,
    # Non-verbal / unable to self-report — assume high acuity
    'non-verbal': 8, 'nonverbal': 8, 'unresponsive': 8, 'non verbal': 8,
    'ett': 8,                      # intubated; cannot self-report pain
    # Maximum
    'worst pain': 10, 'excruciating': 10, '10/10': 10, 'unbearable': 10,
    'critical': 10, 'max': 10, 'crit': 10,
}

# Prehospital values are field-entered before ED assessment; not clinically mappable
PREHOSPITAL = {'prehosp', 'prehospital', 'pre-hospital', 'pre-hosp'}

audit_records = []

def convert_pain(val):
    """Convert a raw pain value to an integer score in [0, 10] or NaN."""
    # Step 1: clean numeric already in range
    if is_numeric_pain(val):
        numeric = float(str(val).strip())
        if numeric == int(numeric):
            return int(numeric), 'clean_numeric'
        else:
            return math.ceil(numeric), 'ceiling'

    if val is None or (isinstance(val, float) and math.isnan(val)):
        return np.nan, 'null'

    # Step 2: pre-process — strip surrounding quotes, trailing punctuation, collapse whitespace
    s = str(val).strip()
    s = re.sub(r'^["\']+|["\']+$', '', s)  # strip leading/trailing quotes (e.g. """no""" → no)
    s = re.sub(r'[.]+$', '', s)             # strip trailing dots (e.g. '0..' → '0')
    s = re.sub(r'\s+', ' ', s).strip()
    s_lower = s.lower()

    # Step 3: out-of-range numerics → cap at 10
    # Handles '13' (426 rows), '12', '11', '15', '95', '100', '180'
    try:
        numeric = float(s_lower)
        if numeric > 10:
            return 10, 'capped_at_10'
        elif numeric >= 0:
            # valid after pre-processing (e.g. '0..' → '0' → 0)
            return math.ceil(numeric) if numeric != int(numeric) else int(numeric), 'ceiling'
    except ValueError:
        pass

    # Step 4: prehospital
    if s_lower in PREHOSPITAL:
        return np.nan, 'prehospital'

    # Step 5: X/Y slash format (e.g. '3/10' → 3, '8/10' → 8, '10/' → 10)
    slash_match = re.match(r'^(\d+)/(\d*)$', s_lower)
    if slash_match:
        numerator = int(slash_match.group(1))
        return min(numerator, 10), 'x_of_y'

    # Step 6: dash range strings (e.g. '5-6' → 6)
    range_match = re.match(r'^(\d+)-(\d+)$', s_lower)
    if range_match:
        high = int(range_match.group(2))
        return min(high, 10), 'range_high'

    # Step 7: semantic map (applied to pre-processed lowercase string)
    if s_lower in PAIN_MAP:
        return PAIN_MAP[s_lower], 'semantic_map'

    return np.nan, 'unrecognised'

scores = []
rules  = []
for val in df['pain']:
    score, rule = convert_pain(val)
    scores.append(score)
    rules.append(rule)
    if rule != 'clean_numeric':
        audit_records.append({'raw_value': val, 'rule': rule, 'assigned_score': score})

df['pain'] = scores
print(f'Conversion complete. Total rows processed: {len(df)}')
print(f'\nRule breakdown:')
from collections import Counter
rule_counts = Counter(rules)
for rule, count in sorted(rule_counts.items(), key=lambda x: -x[1]):
    print(f'  {rule:<20} {count:>5}')

Conversion complete. Total rows processed: 9149

Rule breakdown:
  clean_numeric         8123
  capped_at_10           435
  semantic_map           265
  null                   260
  unrecognised            26
  range_high              18
  ceiling                 13
  prehospital              6
  x_of_y                   3


In [78]:
# Report unrecognised pain values (assigned NaN)

audit_df = pd.DataFrame(audit_records)

unrecognised = audit_df[audit_df['rule'] == 'unrecognised']
if len(unrecognised) > 0:
    freq = unrecognised['raw_value'].value_counts()
    print('=== UNRECOGNISED pain values (assigned NaN + pain_missing=1) ===')
    for val, cnt in freq.items():
        print(f'  {repr(val):40s}  freq={cnt}')
    # Ensure pain_missing is set to 1 for all unrecognised rows
    unrecognised_mask = pd.Series(rules, index=df.index) == 'unrecognised'
    df.loc[unrecognised_mask, 'pain_missing'] = 1
    print(f'\nTotal unrecognised entries: {len(unrecognised)} -> assigned NaN, pain_missing=1')
else:
    print('No unrecognised pain values -- all non-numeric values were handled.')

prehospital_mask = pd.Series(rules, index=df.index) == 'prehospital'
df.loc[prehospital_mask, 'pain_missing'] = 1
print(f'Prehospital values -> NaN + pain_missing=1: {prehospital_mask.sum()} rows')


=== UNRECOGNISED pain values (assigned NaN + pain_missing=1) ===
  'c'                                       freq=7
  'o'                                       freq=3
  'yes'                                     freq=3
  '+'                                       freq=2
  'pain'                                    freq=2
  '___'                                     freq=1
  '++'                                      freq=1
  'Critcal'                                 freq=1
  'unbale'                                  freq=1
  'unalb e'                                 freq=1
  'asleep'                                  freq=1
  'Refusing'                                freq=1
  'Yes'                                     freq=1
  'refused '                                freq=1

Total unrecognised entries: 26 -> assigned NaN, pain_missing=1
Prehospital values -> NaN + pain_missing=1: 6 rows


In [79]:
# Full audit table: ALL non-numeric raw values and their assigned scores

print('=== Full audit: non-numeric raw pain values and assigned scores ===')
print(f'{"Raw Value":<35} {"Rule":<16} {"Assigned Score":<16} {"Frequency"}')
print('-' * 80)

audit_summary = (
    audit_df[audit_df['rule'] != 'null']
    .groupby(['raw_value', 'rule', 'assigned_score'], dropna=False)
    .size()
    .reset_index(name='frequency')
    .sort_values('frequency', ascending=False)
)

for _, row in audit_summary.iterrows():
    score_str = str(int(row['assigned_score'])) if pd.notna(row['assigned_score']) else 'NaN'
    print(f"  {str(row['raw_value']):<33} {row['rule']:<16} {score_str:<16} {row['frequency']}")

print(f"\nTotal non-numeric entries audited: {len(audit_df[audit_df['rule'] != 'null'])}")


=== Full audit: non-numeric raw pain values and assigned scores ===
Raw Value                           Rule             Assigned Score   Frequency
--------------------------------------------------------------------------------
  13                                capped_at_10     10               426
  unable                            semantic_map     0                79
  Critical                          semantic_map     10               47
  uta                               semantic_map     0                35
  ua                                semantic_map     0                23
  UTA                               semantic_map     0                21
  critical                          semantic_map     10               8
  c                                 unrecognised     NaN              7
  UA                                semantic_map     0                6
  unable                            semantic_map     0                5
  prehosp                           prehospi

In [80]:
# Step 6: Final validation
# Assert all non-null pain values are in [0, 10]. Print distribution.

pain_nonnull = df['pain'].dropna()

assert pain_nonnull.between(0, 10).all(), (
    f'Out-of-range pain values detected:\n'
    f'{pain_nonnull[~pain_nonnull.between(0, 10)].value_counts()}'
)
print('Assertion passed: all non-null pain values are in [0, 10].')
print(f'\npain dtype: {df["pain"].dtype}')
print(f'Null count after mapping: {df["pain"].isna().sum()} ({df["pain"].isna().mean()*100:.1f}%)')
print(f'pain_missing=1 count (final): {df["pain_missing"].sum()}')
print('\nPain score value_counts (sorted by score):')
print(df['pain'].value_counts(dropna=False).sort_index().to_string())


Assertion passed: all non-null pain values are in [0, 10].

pain dtype: float64
Null count after mapping: 292 (3.2%)
pain_missing=1 count (final): 1014

Pain score value_counts (sorted by score):
pain
0.0     2860
1.0      128
2.0      313
3.0      372
4.0      409
5.0      654
6.0      576
7.0      698
8.0      911
9.0      451
10.0    1485
NaN      292


## Section 9 — Clean: `arrival_transport`

Standardize transport mode categories to a controlled vocabulary: `WALK IN`, `AMBULANCE`, `HELICOPTER`, `UNKNOWN`.

`OTHER` and `UNKNOWN` both represent non-specific transport information and are merged.


In [81]:
# Step 1: Uppercase and strip whitespace for consistent comparison
df['arrival_transport'] = df['arrival_transport'].str.upper().str.strip()

# Step 2: Merge OTHER into UNKNOWN — both represent unspecified transport modes
df['arrival_transport'] = df['arrival_transport'].replace('OTHER', 'UNKNOWN')

# Step 3: Validate only expected categories remain
expected_categories = {'WALK IN', 'AMBULANCE', 'HELICOPTER', 'UNKNOWN'}
actual_categories = set(df['arrival_transport'].dropna().unique())
unexpected = actual_categories - expected_categories

if unexpected:
    print(f'Unexpected values found: {unexpected}')
else:
    print('Validation passed -- only expected categories present.')

print('\narrival_transport value counts:')
print(df['arrival_transport'].value_counts())


Validation passed -- only expected categories present.

arrival_transport value counts:
arrival_transport
WALK IN       4471
AMBULANCE     4262
UNKNOWN        394
HELICOPTER      22
Name: count, dtype: int64


## Section 10 — Drop Null / Empty Rows

Remove rows that violate the rule: **every included cell must have a real value — no NaN, no empty string.**

| Column | Drop condition | Reason |
|--------|---------------|--------|
| `chiefcomplaint` | `isna()` | Core text feature; cannot impute |
| `HPI` | `isna()` | Core text feature; cannot impute |
| `past_medical_history` | `== ""` | Empty string = no usable PMH signal |
| `pain` | `isna()` | Unresolvable pain value; pain_missing flag already set |
| `initial_vitals` | `isna()` | Row has zero vital sign data |

Rows may trigger multiple conditions; each is reported independently before the combined drop.

In [82]:
before = len(df)

# Build individual drop masks — report each separately before combining
cc_mask    = df['chiefcomplaint'].isna()
hpi_mask   = df['HPI'].isna()
pmh_mask   = df['past_medical_history'] == ''
pain_mask  = df['pain'].isna()
vitals_mask = df['initial_vitals'].isna()

drop_mask = cc_mask | hpi_mask | pmh_mask | pain_mask | vitals_mask

print('Drop breakdown (rows may overlap):')
print(f'  chiefcomplaint null       : {cc_mask.sum():>5}')
print(f'  HPI null                  : {hpi_mask.sum():>5}')
print(f'  past_medical_history ""   : {pmh_mask.sum():>5}')
print(f'  pain null                 : {pain_mask.sum():>5}')
print(f'  initial_vitals null       : {vitals_mask.sum():>5}')
print(f'  ─────────────────────────────────')
print(f'  Unique rows to drop       : {drop_mask.sum():>5}')

df = df[~drop_mask].reset_index(drop=True)

after = len(df)
print(f'\nDropped {before - after} rows. Final row count: {after}')
print(f'Final shape: {df.shape}')

# Verify no nulls or empty strings remain in any required column
required_cols = ['chiefcomplaint', 'HPI', 'past_medical_history', 'pain', 'initial_vitals']
for col in required_cols:
    n_null  = df[col].isna().sum()
    n_empty = (df[col] == '').sum() if df[col].dtype == object else 0
    status  = '✓' if (n_null == 0 and n_empty == 0) else '✗'
    print(f'  {status}  {col:<30}  null={n_null}  empty={n_empty}')

Drop breakdown (rows may overlap):
  chiefcomplaint null       :     7
  HPI null                  :     1
  past_medical_history ""   :   292
  pain null                 :   292
  initial_vitals null       :   370
  ─────────────────────────────────
  Unique rows to drop       :   766

Dropped 766 rows. Final row count: 8383
Final shape: (8383, 21)
  ✓  chiefcomplaint                  null=0  empty=0
  ✓  HPI                             null=0  empty=0
  ✓  past_medical_history            null=0  empty=0
  ✓  pain                            null=0  empty=0
  ✓  initial_vitals                  null=0  empty=0


## Section 11 — Save Cleaned Dataset

Persist the fully cleaned dataframe. Print final shape, null counts, and a 5-row sample.


In [83]:
OUTPUT_KEY    = "Data_Output/consolidated_dataset_cleaned.csv"
OUTPUT_S3_URI = f"s3://{S3_BUCKET}/{OUTPUT_KEY}"

# Serialize to an in-memory buffer and upload to S3
buffer = io.BytesIO()
df.to_csv(buffer, index=False)
buffer.seek(0)

s3.put_object(
    Bucket=S3_BUCKET,
    Key=OUTPUT_KEY,
    Body=buffer.getvalue(),
    ContentType="text/csv",
)

print(f"Saved to    : {OUTPUT_S3_URI}")
print(f"Final shape : {df.shape}")

print("\nNull counts per column (sorted descending):")
print(df.isnull().sum().sort_values(ascending=False).to_string())

print("\nSample (5 rows):")
df.head(5)

Saved to    : s3://ed-triage-capstone-group7/Data_Output/consolidated_dataset_cleaned.csv
Final shape : (8383, 21)

Null counts per column (sorted descending):
temp_f                  169
resp_rate               115
spo2                     94
dbp                      48
sbp                      32
heart_rate               18
race                      0
gender                    0
age                       0
vitals_out_of_range       0
vitals_any_missing        0
stay_id                   0
triage                    0
arrival_transport         0
pain                      0
patient_info              0
initial_vitals            0
past_medical_history      0
HPI                       0
chiefcomplaint            0
pain_missing              0

Sample (5 rows):


,stay_id,triage,chiefcomplaint,HPI,past_medical_history,initial_vitals,patient_info,pain,arrival_transport,temp_f,...,resp_rate,spo2,sbp,dbp,vitals_any_missing,vitals_out_of_range,age,gender,race,pain_missing
0,32822973,2,LETHARGY/shortness of breath,Patient is an [REDACTED] year-old patient with...,Anemia\nBorderline cholesterol\nC. Diff\nFlatu...,"Temperature: 100.2, Heartrate: 95.0, resprate:...","Gender: Female, Race: WHITE, Age: 80",0.0,AMBULANCE,100.2,...,28.0,100.0,99.0,47.0,False,False,80,Female,WHITE,1
1,31591237,2,SEVERE abdominal PAIN,[REDACTED] year old male with history to intes...,1. Sarcoidosis<comma> dx skin bx: intestinal &...,"Temperature: 97.8, Heartrate: 87.0, resprate: ...","Gender: Male, Race: WHITE, Age: 69",10.0,WALK IN,97.8,...,26.0,99.0,119.0,63.0,False,False,69,Male,WHITE,0
2,36819102,3,R Leg pain,[REDACTED] w Rt AK pop to [REDACTED] bypass wi...,PMH: DVT R pop v ([REDACTED])<comma> asthma<co...,"Temperature: 98.7, Heartrate: 93.0, resprate: ...","Gender: Female, Race: WHITE, Age: 66",5.0,WALK IN,98.7,...,16.0,99.0,166.0,97.0,False,False,66,Female,WHITE,0
3,34503980,3,L KNEE PAIN,[REDACTED] with pmhx of chronic LBP [REDACTED]...,1. Hypertension\n2. Hypothyroidism<comma> stat...,"Temperature: 97.0, Heartrate: 75.0, resprate: ...","Gender: Female, Race: WHITE, Age: 62",5.0,AMBULANCE,97.0,...,18.0,97.0,161.0,76.0,False,False,62,Female,WHITE,0
4,31176964,3,abdominal pain,[REDACTED] HIV cirrhosis alcoholism who presen...,- HIV\n- ? Schizoaffective disorder\n- Alcohol...,"Temperature: 97.1, Heartrate: 100.0, resprate:...","Gender: Male, Race: WHITE, Age: 52",8.0,AMBULANCE,97.1,...,18.0,98.0,130.0,98.0,False,False,52,Male,WHITE,0
